# Lab 3 · Why Transformers beat RNNs and LSTMs

**Time:** 40 min · **Work in teams of 3** · **You will:** give the *same* task to an RNN, an LSTM and a Transformer, using the *same* data and the *same* training loop, and see where the first two break and why the third does not. Then look inside a real Transformer (BERT) to see which words a word pays attention to, and measure why Transformers train and run faster.

**Idea:** an RNN and an LSTM read a sentence one word at a time and carry everything in a single small memory. The longer the sentence, the more that memory has to forget. A Transformer lets every word look directly at every other word.

**The running example:**
> *"I went to the **bank** to deposit a cheque."* — a financial bank
> *"I sat on the **bank** and watched the river flow."* — a river bank

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**. Training takes about 3 minutes on the GPU. Before each experiment, agree in your team what you expect and write it down. Then read the result, and change the `🔧` values.*

| Part | What you will see |
|---|---|
| 1 | The "bank" problem: accuracy of RNN, LSTM and Transformer as the clue moves further away |
| 2 | Inside a real Transformer (BERT): what `bank` attends to |
| 3 | Repeating a Smart-TV request word for word: RNN nonsense, LSTM breaks, Transformer exact |
| 4 | Why Transformers won *now*: parallel computation, measured |

In [ ]:
import sys, os
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import check

import torch, torch.nn as nn, random, time, math, itertools, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()

DEV = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = 800                                   # training budget — identical for every model (GPU just finishes faster)
torch.manual_seed(0); random.seed(0)
print(f"Device: {DEV}   |   training steps per model: {STEPS}")
if DEV == "cuda": print("GPU:", torch.cuda.get_device_name(0))

---
# Part 1 — The "bank" problem, three models, one codebase

## 3.1 The data

We generate sentences where **`bank`** must be disambiguated:

* a **clue word** decides the sense — `deposit, cheque, loan…` → *finance*; `river, water, boat…` → *river*
* the clue is placed **`d` words away** from `bank` (d = 0 … 34)
* to make it realistic, every sentence also contains a **distractor** of the *opposite* sense, always **farther away** from `bank` than the true clue — like a real paragraph that mentions both a bank and a river.

The rule a model has to learn is the human rule: *"the word closest to `bank` tells you what it means."*
That requires relating two words **across a distance while ignoring a third** — the exact thing self-attention is built for.

In [ ]:
FIN  = ["deposit","cheque","cash","loan","account","withdraw","teller","savings","atm","interest"]
RIV  = ["river","water","fish","flow","mud","boat","shore","fishing","stream","current"]
FILL = ("and then the a very quite slowly quickly later again there here old new big small walked looked "
        "day morning evening friend man woman child dog road town village city sun sky tree car bus "
        "we they he she it was were with near far after before some many "
        "i went to on sat watched away long walk found asked for").split()

vocab = ["<pad>", "bank"] + FIN + RIV + FILL
stoi  = {w: i for i, w in enumerate(vocab)}
PAD, BANK = 0, 1
MAXLEN, DMAX = 80, 34                      # sentence length cap, max clue distance
LABEL = {1: "FINANCE", 0: "RIVER"}

def filler(n): return [random.choice(FILL) for _ in range(n)]

def make_sentence(d, label=None):
    "True clue at distance d from 'bank'; opposite-sense distractor always farther away."
    if label is None: label = random.randint(0, 1)
    clue       = random.choice(FIN if label else RIV)
    distractor = random.choice(RIV if label else FIN)
    e = d + random.randint(2, 6)                                   # distractor distance > clue distance
    order = random.choice("ABCD")                                  # where things sit in the sentence
    if   order == "A": toks = [clue] + filler(d) + ["bank"] + filler(e) + [distractor]
    elif order == "B": toks = [distractor] + filler(e) + [clue] + filler(d) + ["bank"]
    elif order == "C": toks = ["bank"] + filler(d) + [clue] + filler(e) + [distractor]
    else:              toks = [distractor] + filler(e) + ["bank"] + filler(d) + [clue]
    toks = filler(random.randint(0, 2)) + toks + filler(random.randint(0, 2))
    return toks, label, order

def encode(toks):
    ids = [stoi[t] for t in toks][:MAXLEN]
    return ids + [PAD] * (MAXLEN - len(ids))

def make_dataset(n, dmax=DMAX):
    X, Y, D, O = [], [], [], []
    for _ in range(n):
        d = random.randint(0, dmax)
        toks, y, o = make_sentence(d)
        X.append(encode(toks)); Y.append(y); D.append(d); O.append(o)
    return torch.tensor(X), torch.tensor(Y), torch.tensor(D), O

Xtr, Ytr, Dtr, Otr = make_dataset(8000)
Xte, Yte, Dte, Ote = make_dataset(3000)

print("Three example sentences (clue in CAPITALS, distractor in [brackets]):\n")
for d in (2, 12, 30):
    toks, y, o = make_sentence(d)
    pretty = " ".join(t.upper() if (t in FIN and y) or (t in RIV and not y) else f"[{t}]" if t in FIN + RIV else t for t in toks)
    print(f"d={d:2d}  → {LABEL[y]:8s} | {pretty}\n")

## 3.2 The three models

All three share the **same embedding layer and the same output layer**. Only the middle differs.

```
tokens → Embedding → [ RNN | LSTM | Transformer ] → Linear → FINANCE / RIVER
```

* **RNN** — reads left to right, one word at a time; the whole past is squeezed into one hidden vector `h`.
* **LSTM** — same, but with gates (forget / input / output) and a *cell state* so information survives longer.
* **Transformer** — every word looks at **every other word at once** through *self-attention*:

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

The Transformer below is written out in full — `Wq`, `Wk`, `Wv`, the dot product, the softmax — so we can **read the attention weights back** afterwards.

*Position:* attention by itself is order-blind, so Transformers add position information. The original paper adds sinusoidal position vectors to the embeddings; BERT/GPT-2 learn a position table; this toy uses a **distance bias inside attention** (the ALiBi idea used by BLOOM and MPT) — nearer words get a head start. Same purpose, and it generalises to longer sentences.

In [ ]:
class RNNClassifier(nn.Module):
    "RNN or LSTM: read the sentence word by word, classify from the FINAL hidden state."
    def __init__(self, cell="rnn", d=64):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), d, padding_idx=PAD)
        self.rnn = (nn.RNN if cell == "rnn" else nn.LSTM)(d, d, batch_first=True)
        self.out = nn.Linear(d, 2)
    def forward(self, x):
        lengths = (x != PAD).sum(1)
        h, _ = self.rnn(self.emb(x))                              # h: (batch, L, d) — one step at a time
        last = h[torch.arange(x.size(0)), lengths - 1]            # hidden state after the LAST word
        return self.out(last)


class SelfAttention(nn.Module):
    "One multi-head self-attention layer, written out step by step."
    def __init__(self, d, heads):
        super().__init__()
        self.h, self.dk = heads, d // heads
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
        self.slope = nn.Parameter(torch.linspace(0.1, 1.0, heads))   # learned "nearer words matter more" bias per head
    def forward(self, x, pad_mask):
        B, L, d = x.shape
        Q = self.Wq(x).view(B, L, self.h, self.dk).transpose(1, 2)   # Query : "what am I looking for?"
        K = self.Wk(x).view(B, L, self.h, self.dk).transpose(1, 2)   # Key   : "what do I contain?"
        V = self.Wv(x).view(B, L, self.h, self.dk).transpose(1, 2)   # Value : "what do I hand over?"
        scores = Q @ K.transpose(-1, -2) / math.sqrt(self.dk)        # every word × every word  (B, h, L, L)
        pos  = torch.arange(L, device=x.device)
        dist = (pos[None, :] - pos[:, None]).abs().float()           # |i − j|
        scores = scores - self.slope.view(1, self.h, 1, 1) * dist    # positional bias (ALiBi-style)
        scores = scores.masked_fill(pad_mask[:, None, None, :], float("-inf"))
        attn = scores.softmax(-1)                                    # each row sums to 1  ← the attention weights
        out  = (attn @ V).transpose(1, 2).reshape(B, L, d)           # weighted sum of Values
        return self.Wo(out), attn


class TransformerBlock(nn.Module):
    "Attention → Add & Norm → Feed-forward → Add & Norm  (one encoder layer)"
    def __init__(self, d, heads):
        super().__init__()
        self.attn, self.ln1, self.ln2 = SelfAttention(d, heads), nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x, pad_mask):
        a, attn = self.attn(self.ln1(x), pad_mask); x = x + a          # residual connection
        x = x + self.ffn(self.ln2(x))                                   # residual connection
        return x, attn


class TransformerClassifier(nn.Module):
    "Embedding + position → N encoder blocks → classify from the vector sitting at the word 'bank'."
    def __init__(self, d=64, heads=4, layers=2):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), d, padding_idx=PAD)
        # No separate positional-encoding table here: word order enters through the distance bias
        # inside SelfAttention (the ALiBi idea, used by BLOOM/MPT). It also works for longer sentences than seen in training.
        self.blocks = nn.ModuleList([TransformerBlock(d, heads) for _ in range(layers)])
        self.out = nn.Linear(d, 2)
    def forward(self, x, return_attn=False):
        pad = x == PAD
        h = self.emb(x)
        attns = []
        for blk in self.blocks:
            h, a = blk(h, pad); attns.append(a)
        bank_pos = (x == BANK).float().argmax(1)
        logits = self.out(h[torch.arange(x.size(0)), bank_pos])       # read the contextualised 'bank' vector
        return (logits, attns) if return_attn else logits

for name, m in [("RNN", RNNClassifier("rnn")), ("LSTM", RNNClassifier("lstm")), ("Transformer", TransformerClassifier())]:
    print(f"{name:12s} parameters: {sum(p.numel() for p in m.parameters()):,}")

## 3.3 One training loop for all three

Nothing model-specific here — this is the point: **the code around the model is identical**.

**Predict first (write it down):** at which distance between `bank` and its clue will each model start to fail? Which model will *not* fail?

In [ ]:
def train(model, X, Y, steps=STEPS, bs=64, lr=2e-3):
    model.to(DEV).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    t0 = time.time()
    for s in range(steps):
        i  = torch.randint(0, len(X), (bs,))
        xb = X[i]; xb = xb[:, : (xb != PAD).sum(1).max()]              # trim padding
        loss = nn.functional.cross_entropy(model(xb.to(DEV)), Y[i].to(DEV))
        opt.zero_grad(); loss.backward(); opt.step()
    return time.time() - t0

@torch.no_grad()
def evaluate(model, X, Y, D, O):
    model.eval(); pred = model(X.to(DEV)).argmax(1).cpu(); model.train()
    ok = (pred == Y)
    buckets = [(0, 2), (3, 8), (9, 16), (17, 24), (25, DMAX)]
    by_d = {f"{a}-{b}": ok[(D >= a) & (D <= b)].float().mean().item() for a, b in buckets}
    return ok.float().mean().item(), by_d, pred

models  = {"RNN": RNNClassifier("rnn"), "LSTM": RNNClassifier("lstm"), "Transformer": TransformerClassifier()}
results = {}
for name, m in models.items():
    secs = train(m, Xtr, Ytr, lr=1e-3 if name == "Transformer" else 2e-3)
    acc, by_d, pred = evaluate(m, Xte, Yte, Dte, Ote)
    results[name] = (acc, by_d, pred)
    print(f"{name:12s} trained in {secs:5.1f}s   overall accuracy = {acc:.1%}")

In [ ]:
far = {n: results[n][1]["25-34"] for n in results}
check(far["Transformer"] > 0.9 and far["Transformer"] > far["RNN"] + 0.2,
      f"With the clue 25 to 34 words away: RNN {far['RNN']:.0%}, LSTM {far['LSTM']:.0%}, Transformer {far['Transformer']:.0%} (chance is 50%). "
      "Same code, same data, same training. Only the model block changed.",
      "the Transformer did not clearly beat the RNN at long distance - re-run the training cell (results vary a little between runs).")

## 3.4 Accuracy vs. distance between `bank` and its clue

Chance level is 50 %. Watch what happens as the clue moves farther away.

In [ ]:
buckets = list(results["RNN"][1].keys())
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.26
for k, (name, (acc, by_d, _)) in enumerate(results.items()):
    ax.bar([i + (k - 1) * w for i in range(len(buckets))], [by_d[b] * 100 for b in buckets], w, label=f"{name} ({acc:.0%})")
ax.axhline(50, ls="--", c="grey", lw=1); ax.text(len(buckets) - 0.5, 51, "chance", color="grey")
ax.set_xticks(range(len(buckets))); ax.set_xticklabels([f"{b} words" for b in buckets])
ax.set_xlabel("distance between 'bank' and its clue"); ax.set_ylabel("accuracy %"); ax.set_ylim(0, 105)
ax.set_title("Same code, same data, same training budget — only the model block changed"); ax.legend()
plt.tight_layout(); plt.show()

print(f"{'distance':>10s} | " + " | ".join(f"{n:>11s}" for n in results))
for b in buckets:
    print(f"{b:>10s} | " + " | ".join(f"{results[n][1][b]:>10.1%}" for n in results))

### 🔧 Try it 3.4 — Pick your own distance
The chart groups distances into buckets. Pick **one exact distance** `d` and test all three models on 300 fresh sentences.

🔧 **Try:** `d = 2`, then `d = 12`, then `d = 30`. Find the distance where the LSTM stops being much better than guessing.

In [ ]:
d = 20          # 🔧 words between 'bank' and its clue (0 to 34)
random.seed(1)
fresh = [make_sentence(d) for _ in range(300)]
Xf = torch.tensor([encode(t) for t, _, _ in fresh]); Yf = torch.tensor([y for _, y, _ in fresh])
for name, m in models.items():
    m.eval()
    with torch.no_grad():
        acc = (m(Xf.to(DEV)).argmax(1).cpu() == Yf).float().mean().item()
    m.train()
    print(f"{name:12s} at distance {d:2d}: {acc:.0%}")

## 3.5 Where exactly do the RNN and LSTM go wrong?

Concrete sentences the RNN and/or LSTM get **wrong** and the Transformer gets **right**.

In [ ]:
def decode(row):
    toks = [vocab[i] for i in row.tolist() if i != PAD]
    return " ".join(t.upper() if t in FIN + RIV else t for t in toks)

wrong = [(i) for i in range(len(Xte))
         if results["Transformer"][2][i] == Yte[i] and (results["RNN"][2][i] != Yte[i] or results["LSTM"][2][i] != Yte[i])]
print(f"{len(wrong)} of {len(Xte)} test sentences: Transformer right, RNN and/or LSTM wrong.\n")
for i in wrong[:6]:
    print(decode(Xte[i]))
    print(f"   truth={LABEL[Yte[i].item()]:8s}  RNN={LABEL[results['RNN'][2][i].item()]:8s}  "
          f"LSTM={LABEL[results['LSTM'][2][i].item()]:8s}  Transformer={LABEL[results['Transformer'][2][i].item()]}\n")

## 3.6 Look inside: what does `bank` attend to?

This is the classic illustration — *Attention(bank → deposit) ≈ 0.45, Attention(bank → cheque) ≈ 0.40* — except here the numbers come out of a model **we just trained**. We take the attention row for `bank` (averaged over heads, last layer) and plot it.

In [ ]:
@torch.no_grad()
def show_bank_attention(sentence_tokens, layer=-1):
    m = models["Transformer"].eval()
    x = torch.tensor([encode(sentence_tokens)])[:, :len(sentence_tokens)].to(DEV)
    logits, attns = m(x, return_attn=True)
    i = sentence_tokens.index("bank")
    row = attns[layer][0].mean(0)[i].cpu()                       # average over heads, row for 'bank'
    top = sorted(zip(sentence_tokens, row.tolist()), key=lambda t: -t[1])[:3]
    fig, ax = plt.subplots(figsize=(max(6, 0.45 * len(sentence_tokens)), 2.6))
    ax.bar(range(len(sentence_tokens)), row, color=["#d62728" if t in FIN + RIV else "#1f77b4" for t in sentence_tokens])
    ax.set_xticks(range(len(sentence_tokens))); ax.set_xticklabels(sentence_tokens, rotation=60, ha="right")
    ax.set_ylabel("attention from 'bank'")
    ax.set_title(f"prediction = {LABEL[logits.argmax().item()]}   |   top: " + ", ".join(f"{t} {w:.2f}" for t, w in top))
    plt.tight_layout(); plt.show()

show_bank_attention("i went to the bank to deposit a cheque".split())
show_bank_attention("i sat on the bank and watched the river flow".split())
show_bank_attention("the river was far away and after a long walk we found the bank and asked the teller for cash".split())

> **Why this matters:** the RNN had to *carry* the clue through every intermediate word and hope it survived. The Transformer simply **looks straight at it** — one dot product, regardless of distance — and the distractor gets a near-zero weight.

### Optional stretch: make the sentences twice as long (GPU recommended)
Optional. With 160-word sentences the LSTM collapses to chance even with the full training budget; the Transformer does not.

In [ ]:
RUN_STRETCH = False          # set True to run (≈ 1–2 min on GPU)
if RUN_STRETCH:
    MAXLEN_OLD, DMAX_OLD = MAXLEN, DMAX
    MAXLEN, DMAX = 160, 74
    Xl, Yl, Dl, Ol = make_dataset(8000); Xlt, Ylt, Dlt, Olt = make_dataset(2000)
    for name, m in [("LSTM", RNNClassifier("lstm")), ("Transformer", TransformerClassifier())]:
        secs = train(m, Xl, Yl, lr=1e-3 if name == "Transformer" else 2e-3)
        acc, by_d, _ = evaluate(m, Xlt, Ylt, Dlt, Olt)
        print(f"{name:12s} 160-word sentences: accuracy = {acc:.1%}  ({secs:.0f}s)")
    MAXLEN, DMAX = MAXLEN_OLD, DMAX_OLD

---
# Part 2 — Inside a real Transformer: BERT on the same sentences

Our toy model has 100 k parameters. **BERT-base** is the same architecture with 110 M parameters, 12 layers, 12 heads, pretrained on Wikipedia + books. Let's ask it the same question.

## 3.7 Fill in the blank
BERT was trained to predict a masked word from *both sides* of context. Watch it use `deposit` to guess `cheque`, and `flow` to guess `river`.

In [ ]:
from transformers import AutoTokenizer, AutoModel, pipeline
BERT = "bert-base-uncased"
tok   = AutoTokenizer.from_pretrained(BERT)
bert  = AutoModel.from_pretrained(BERT, attn_implementation="eager").to(DEV).eval()   # eager → attention weights available
fill  = pipeline("fill-mask", model=BERT, device=0 if DEV == "cuda" else -1)

for s in ["I went to the bank to deposit a [MASK].",
          "I sat on the bank and watched the [MASK] flow.",
          "I went to the [MASK] to deposit a cheque.",
          "The batsman hit the ball with the [MASK]."]:
    print(f"{s:55s} → " + ", ".join(f"{r['token_str']} ({r['score']:.0%})" for r in fill(s, top_k=3)))

## 3.8 The long-range dependency: *"amazing"* must reach back to *"movie"*

> *"The movie that I watched last month during my vacation in London was amazing."*

Twelve words separate the thing being praised from the praise. An RNN has to carry `movie` through twelve overwrites. BERT simply looks back.

In [ ]:
long_s = "The movie that I watched last month during my vacation in London was amazing."
enc = tok(long_s, return_tensors="pt").to(DEV)
with torch.no_grad(): out = bert(**enc, output_attentions=True)
ltoks = tok.convert_ids_to_tokens(enc["input_ids"][0]); i = ltoks.index("amazing")
row = out.attentions[3][0].mean(0)[i].cpu().clone()               # layer 4, heads averaged
for j, t in enumerate(ltoks):
    if t in ("[CLS]", "[SEP]", ".") or j == i: row[j] = 0
row /= row.sum()
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.bar(range(len(ltoks)), row, color=["#d62728" if t == "movie" else "#1f77b4" for t in ltoks])
ax.set_xticks(range(len(ltoks))); ax.set_xticklabels(ltoks, rotation=60, ha="right"); ax.set_ylabel("attention from 'amazing'")
top = sorted(zip(ltoks, row.tolist()), key=lambda t: -t[1])[:3]
ax.set_title("amazing → " + ", ".join(f"{t} {w:.2f}" for t, w in top) + f"   (distance: {i - ltoks.index('movie')} words)")
plt.tight_layout(); plt.show()

masked = "The [MASK] that I watched last month during my vacation in London was amazing."
print(masked, "→", ", ".join(f"{r['token_str']} ({r['score']:.0%})" for r in fill(masked, top_k=3)))

> `amazing` puts its largest weight on `movie`, twelve words back, and when `movie` is hidden BERT recovers it from `watched … amazing`. The connection is direct — no chain to survive.

## 3.9 Static vs. contextual vectors — the one number that explains everything

Word2Vec / GloVe give **one** vector per word: `bank` is the same vector in both sentences.
A Transformer gives a **different** vector for `bank` in each sentence, because attention mixes in the neighbours.

We measure cosine similarity of the `bank` vector **before** the Transformer layers (static input embedding) and **after** them (contextual).

In [ ]:
S = {"finance-1": "I went to the bank to deposit a cheque.",
     "finance-2": "The bank approved my loan and opened a savings account.",
     "river-1"  : "I sat on the bank and watched the river flow.",
     "river-2"  : "The fisherman pulled his boat onto the muddy bank of the stream."}

@torch.no_grad()
def bank_vectors(sent):
    enc = tok(sent, return_tensors="pt").to(DEV)
    out = bert(**enc, output_hidden_states=True, output_attentions=True)
    i   = enc["input_ids"][0].tolist().index(tok.convert_tokens_to_ids("bank"))
    return {"static": out.hidden_states[0][0, i].cpu(), "contextual": out.hidden_states[-1][0, i].cpu(),
            "attn": [a[0].cpu() for a in out.attentions], "tokens": tok.convert_ids_to_tokens(enc["input_ids"][0])}

V   = {k: bank_vectors(v) for k, v in S.items()}
cos = lambda a, b: nn.functional.cosine_similarity(a, b, dim=0).item()

print(f"{'pair':28s} {'STATIC (word2vec-style)':>25s} {'CONTEXTUAL (after Transformer)':>32s}")
for a, b in itertools.combinations(S, 2):
    same = "same sense " if a.split('-')[0] == b.split('-')[0] else "DIFF sense "
    print(f"{same}{a:>10s} vs {b:<10s} {cos(V[a]['static'], V[b]['static']):>25.3f} {cos(V[a]['contextual'], V[b]['contextual']):>32.3f}")

> **Read the table:** on the static side `bank` in the cheque sentence and `bank` in the river sentence are **identical (1.000)** — the model cannot tell them apart. After self-attention the two are far apart (~0.45), while the two *finance* banks stay close (~0.80). **That is word-sense disambiguation, learned, with no labels.**

## 3.10 The attention map — which words does `bank` look at?

In [ ]:
def attention_row(v, word="bank", layer=0):
    "Attention from `word` to every other token, averaged over heads, special tokens & self removed, renormalised."
    toks = v["tokens"]; i = toks.index(word)
    row = v["attn"][layer].mean(0)[i].clone()
    for j, t in enumerate(toks):
        if t in ("[CLS]", "[SEP]") or j == i: row[j] = 0
    return toks, row / row.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
for ax, key in zip(axes, ["finance-1", "river-1"]):
    toks, row = attention_row(V[key], layer=0)
    ax.bar(range(len(toks)), row, color=["#d62728" if t in ("deposit", "che", "##que", "river", "flow") else "#1f77b4" for t in toks])
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=60, ha="right")
    top = sorted(zip(toks, row.tolist()), key=lambda t: -t[1])[:2]
    ax.set_title(f"'{S[key]}'\nbank → " + ", ".join(f"{t} {w:.2f}" for t, w in top)); ax.set_ylabel("attention (layer 1)")
plt.tight_layout(); plt.show()

# full L×L matrix for the finance sentence, one layer, heads averaged
toks = V["finance-1"]["tokens"]; A = V["finance-1"]["attn"][0].mean(0)
fig, ax = plt.subplots(figsize=(5.5, 5)); im = ax.imshow(A, cmap="Blues")
ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=60, ha="right"); ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks)
ax.set_title("Full self-attention matrix (rows attend to columns)"); plt.colorbar(im); plt.tight_layout(); plt.show()

> Each **row** is a word asking *"who matters for me?"*; each **column** is a word answering. Every word sees every other word in one step — there is no "distance" to travel. That single matrix multiply is what an RNN cannot do.

## 3.11 Bonus — the classic "it" example
*"The animal didn't cross the street because **it** was too tired"* vs *"… because **it** was too wide"*. Which noun should `it` attend to? (Deeper layers of BERT handle this kind of reference; head-averaged so it is indicative, not a coreference system.)

In [ ]:
for s in ["The animal didn't cross the street because it was too tired.",
          "The animal didn't cross the street because it was too wide."]:
    enc = tok(s, return_tensors="pt").to(DEV)
    with torch.no_grad(): out = bert(**enc, output_attentions=True)
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0]); i = toks.index("it")
    row = out.attentions[9][0].mean(0)[i].cpu().clone()          # layer 10, head-averaged
    for j, t in enumerate(toks):
        if t in ("[CLS]", "[SEP]", ".", "'") or j == i: row[j] = 0
    row /= row.sum()
    top = sorted(zip(toks, row.tolist()), key=lambda t: -t[1])[:3]
    print(f"{s}\n   it → " + ", ".join(f"{t} {w:.2f}" for t, w in top) + "\n")

---
# Part 3 — Generation: nonsense vs. exact

A Smart-TV assistant must **repeat the request back** before acting on it (*"Playing Hindi action movies after 2020 on Netflix — correct?"*). To do that a model has to remember every word of the request, in order.

We train three **language models** (next-word predictors) on the same data with the same loop: `<bos> request <sep> request <eos>`. Then we give each a request and let it generate the echo, one word at a time.

**Predict first (write it down):** a request is repeated back word for word. Which model repeats a 4-word request correctly? A 12-word one? A 20-word one?

In [ ]:
WORDS = ("play show find action comedy thriller drama hindi english tamil telugu korean movies series "
         "after before 2018 2020 2022 tom cruise shah rukh khan netflix prime hotstar latest old family kids on not in for").split()
gvocab = ["<pad>", "<bos>", "<sep>", "<eos>"] + WORDS
gstoi  = {w: i for i, w in enumerate(gvocab)}
GPAD, BOS, SEP, EOS = 0, 1, 2, 3
LMAX = 20; SEQ = 2 * LMAX + 3

def make_echo(n):
    req = [random.choice(WORDS) for _ in range(n)]
    ids = [BOS] + [gstoi[w] for w in req] + [SEP] + [gstoi[w] for w in req] + [EOS]
    return ids + [GPAD] * (SEQ - len(ids))
Xg = torch.tensor([make_echo(random.randint(3, LMAX)) for _ in range(20000)])


class RNNLM(nn.Module):
    def __init__(self, cell, d=128):
        super().__init__()
        self.emb = nn.Embedding(len(gvocab), d, padding_idx=GPAD)
        self.rnn = (nn.RNN if cell == "rnn" else nn.LSTM)(d, d, batch_first=True)
        self.out = nn.Linear(d, len(gvocab))
    def forward(self, x): h, _ = self.rnn(self.emb(x)); return self.out(h)


class CausalSelfAttention(nn.Module):
    "Same attention as Part 1, plus a causal mask: a word may only look at words BEFORE it (needed for generation)."
    def __init__(self, d, heads):
        super().__init__(); self.h, self.dk = heads, d // heads; self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
    def forward(self, x):
        B, L, d = x.shape
        q, k, v = (t.transpose(1, 2) for t in self.qkv(x).view(B, L, 3, self.h, self.dk).unbind(2))
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.dk)
        future = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), 1)
        attn = scores.masked_fill(future, float("-inf")).softmax(-1)
        return self.o((attn @ v).transpose(1, 2).reshape(B, L, d))

class GPTBlock(nn.Module):
    def __init__(self, d, heads):
        super().__init__(); self.a = CausalSelfAttention(d, heads); self.l1, self.l2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.f = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x): x = x + self.a(self.l1(x)); return x + self.f(self.l2(x))

class TransformerLM(nn.Module):
    "A miniature GPT."
    def __init__(self, d=128, heads=4, layers=2):
        super().__init__()
        self.emb = nn.Embedding(len(gvocab), d, padding_idx=GPAD); self.pos = nn.Embedding(SEQ, d)
        self.blocks = nn.ModuleList([GPTBlock(d, heads) for _ in range(layers)]); self.ln = nn.LayerNorm(d)
        self.out = nn.Linear(d, len(gvocab))
    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.size(1), device=x.device))
        for b in self.blocks: h = b(h)
        return self.out(self.ln(h))


def train_lm(model, X, steps=STEPS, bs=64, lr=2e-3):
    model.to(DEV).train(); opt = torch.optim.Adam(model.parameters(), lr=lr); t0 = time.time()
    for _ in range(steps):
        xb = X[torch.randint(0, len(X), (bs,))].to(DEV)
        inp, tgt = xb[:, :-1], xb[:, 1:].clone()
        sep = (xb == SEP).float().argmax(1)                              # score only the echo part
        tgt[torch.arange(tgt.size(1), device=DEV)[None, :] < sep[:, None]] = GPAD
        loss = nn.functional.cross_entropy(model(inp).reshape(-1, len(gvocab)), tgt.reshape(-1), ignore_index=GPAD)
        opt.zero_grad(); loss.backward(); opt.step()
    return time.time() - t0

@torch.no_grad()
def echo(model, req):
    "Greedy generation: feed <bos> request <sep>, then predict one word at a time."
    model.eval(); ids = [BOS] + [gstoi[w] for w in req] + [SEP]
    for _ in range(len(req) + 1):
        nxt = model(torch.tensor([ids], device=DEV))[0, -1].argmax().item(); ids.append(nxt)
        if nxt == EOS: break
    model.train(); return [gvocab[i] for i in ids[len(req) + 2:] if i != EOS]

@torch.no_grad()
def exact_match(model, n=100):
    return {L: sum(echo(model, r := [random.choice(WORDS) for _ in range(L)]) == r for _ in range(n)) / n for L in (4, 8, 12, 16, 20)}

lms = {"RNN": RNNLM("rnn"), "LSTM": RNNLM("lstm"), "Transformer": TransformerLM()}
exact = {}
for name, m in lms.items():
    secs = train_lm(m, Xg, steps=2000); exact[name] = exact_match(m)      # same budget for all three
    print(f"{name:12s} trained in {secs:5.1f}s   exact-match by request length: " +
          "  ".join(f"{L}w:{a:.0%}" for L, a in exact[name].items()))

In [ ]:
check(exact["Transformer"][20] >= 0.9 and exact["RNN"][20] < 0.5,
      f"Repeating a 20-word request exactly: RNN {exact['RNN'][20]:.0%}, LSTM {exact['LSTM'][20]:.0%}, Transformer {exact['Transformer'][20]:.0%}. "
      "The RNN gives fluent-looking nonsense: the right vocabulary, the wrong words.",
      "the Transformer did not repeat 20-word requests reliably, or the RNN did unusually well - re-run the training cell.")

In [ ]:
requests = ["play hindi action movies after 2020 on netflix".split(),
            "find latest korean thriller series on prime for family kids after 2022 not old".split(),
            [random.choice(WORDS) for _ in range(18)]]
for req in requests:
    print("REQUEST :", " ".join(req))
    for name, m in lms.items():
        out = echo(m, req); mark = "✅" if out == req else "❌"
        print(f"  {name:12s}{mark} {' '.join(out)}")
    print()

### 🔧 Try it 3.12 — Your own Smart-TV request
Type your own request, using only words the models know (they are listed in `WORDS`). Try a short one, then a long one of 15 to 20 words, and compare what each model says.

🔧 **Try:** make the request longer and longer until the LSTM starts to fail.

In [ ]:
my_request = "play latest hindi comedy series on netflix for family kids".split()     # 🔧 words from WORDS only
unknown = [w for w in my_request if w not in gstoi]
assert not unknown, f"these words are not known to the models: {unknown}. Choose words from: {' '.join(WORDS)}"
print("REQUEST :", " ".join(my_request), f"({len(my_request)} words)")
for name, m in lms.items():
    got = echo(m, my_request)
    print(f"  {name:12s}{'✅' if got == my_request else '❌'} {' '.join(got)}")

> **RNN**: fluent-looking nonsense — the right vocabulary, wrong words. Its single hidden vector was overwritten before it could reproduce the beginning.
> **LSTM**: the gates help — short requests come back right — but long ones get scrambled or truncated.
> **Transformer**: repeats the request **verbatim** because at every step it can attend directly to the word it needs to copy.
>
> This is not a toy phenomenon: it is the reason the original *Attention* paper (Bahdanau 2014) existed, and the reason *Attention Is All You Need* (2017) removed the recurrence altogether.

---
# Part 4 — Why Transformers won *now*: parallelism and scale

An RNN must compute step 1 before step 2 before step 3 … — **L sequential steps** for L words. Self-attention computes **all L×L relationships in one matrix multiply**, which is exactly what a GPU is good at. That means: train on a whole book at once, on thousands of GPUs, on trillions of words. Scale is what turned "a better sequence model" into GPT.

## 3.13 Training-step time vs. sequence length

In [ ]:
@torch.no_grad()
def _sync():
    if DEV == "cuda": torch.cuda.synchronize()

@torch.no_grad()
def step_time(model, L, bs=32, reps=5):
    "Forward-pass time for a batch of `bs` sequences of length L."
    x = torch.randint(4, len(gvocab), (bs, L), device=DEV); model.to(DEV).eval()
    model(x[:, :8]); _sync()                                       # warm-up
    t0 = time.time()
    for _ in range(reps): model(x); _sync()
    return (time.time() - t0) / reps * 1000

class TransformerLMLong(TransformerLM):
    def __init__(self, maxlen): super().__init__(); self.pos = nn.Embedding(maxlen, 128)

lengths = [32, 128, 512] + ([1024] if DEV == "cuda" else [])
rows = {"LSTM": [step_time(RNNLM("lstm"), L) for L in lengths],
        "Transformer": [step_time(TransformerLMLong(max(lengths)), L) for L in lengths]}
print(f"{'sequence length':>16s} | {'LSTM (ms)':>10s} | {'Transformer (ms)':>16s} | sequential steps needed: LSTM vs Transformer")
for i, L in enumerate(lengths): print(f"{L:>16d} | {rows['LSTM'][i]:>10.1f} | {rows['Transformer'][i]:>16.1f} | {L:>6d} vs 1 per layer")

plt.figure(figsize=(6, 3.5))
for k, v in rows.items(): plt.plot(lengths, v, marker="o", label=k)
plt.xlabel("sequence length (tokens)"); plt.ylabel("ms per forward pass"); plt.title(f"batch 32, {DEV.upper()}")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

> **Read the table honestly.** On a tiny model, a GPU's LSTM can be *faster in milliseconds* than a Transformer: the LSTM code is very well optimised, and attention does extra work that grows with the square of the length. So speed at toy size is **not** why Transformers won. Look at what the **last column** says instead: an LSTM of length L needs **L dependent steps**, step 512 cannot start until step 511 is done, so more hardware cannot shorten that chain. The Transformer's work is **one big matrix product per layer** that a GPU can spread over thousands of cores, and that is what lets it grow to billions of parameters and train on trillions of words. *(On a CPU the raw milliseconds can also favour the LSTM, for the same reason.)*

In [ ]:
lstm_growth = rows["LSTM"][-1] / rows["LSTM"][0]
check(lstm_growth > 4 and all(v > 0 for v in rows["Transformer"]),
      f"From {lengths[0]} to {lengths[-1]} tokens the LSTM got {lstm_growth:.0f}x slower, in step with its chain of dependent steps "
      f"(the Transformer: {rows['Transformer'][-1] / rows['Transformer'][0]:.0f}x). At this toy size the LSTM is still faster in milliseconds: "
      "the Transformer's advantage is parallelism and scale, plus the quality you saw in Parts 1 and 3.",
      "the LSTM time did not grow with the sequence length - run the timing cell again.")

## 3.14 The payoff: a pretrained Transformer answering the Smart-TV request
`Qwen2.5-0.5B-Instruct` — a small modern LLM (0.5 B parameters). Same attention block as our 100 k-parameter toy, scaled up ~5000× and pretrained on trillions of tokens.

In [ ]:
gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=0 if DEV == "cuda" else -1)
SYSTEM = "You are the voice assistant of a Smart TV. Be brief and concrete."
for user in ["Explain in two sentences why 'bank' means something different in 'deposit a cheque at the bank' and 'sat on the river bank'.",
             "I want to watch a Hindi action movie released after 2020 on Netflix. Which ones can you suggest?"]:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    print("USER     :", user)
    print("ASSISTANT:", gen(msgs, max_new_tokens=120, do_sample=False)[0]["generated_text"][-1]["content"], "\n")

> **Look closely at the second answer.** The language is fluent and confident — and the film list is wrong (titles from 2016, 1998, or not Hindi at all). The Transformer gives us *fluency and context*; it does **not** give us *facts*. That gap is what **Retrieval-Augmented Generation** closes — the next topic on Day 1.

---
# Summary

| | RNN | LSTM | Transformer |
|---|---|---|---|
| How context reaches a word | passes through **every** intermediate step | same, but gates let it survive longer | **direct** connection, one dot product |
| "bank" task (Part 1) | ≈ chance | partial, worse with distance | ≈ perfect at every distance |
| Echo task (Part 3) | gibberish | short OK, long scrambled | verbatim |
| Long-range dependency | weak (vanishing gradient) | moderate | strong |
| Training | sequential, L steps | sequential, L steps | **parallel**, all tokens at once |
| Scales to billions of params / trillions of tokens | no | no | **yes → LLMs** |
| Interpretability | hidden vector | hidden vector | **attention weights you can plot** |

**One line:** an RNN *remembers*, an LSTM *remembers better*, a Transformer *looks*.

## 🧠 Check your understanding
1. In the "bank" experiment, why does the LSTM do better than the plain RNN, and why does it still fail when the clue is very far away?
2. The Transformer's attention lets `bank` look at the clue directly. What does this cost compared with an LSTM, and when does it start to hurt?
3. A colleague says: "The Transformer repeated the request perfectly, so it understood it." Is that right?
4. In the speed table the LSTM was faster in milliseconds than the Transformer. So why did Transformers still win?

*✍️ Your answer:* 

**Next:** Lab 4 — inside a language model: probabilities, perplexity and attention.